# YOLOv8 实例分割快速入门

本 Notebook 演示 Ultralytics YOLOv8 的实例分割用法：
- 加载预训练分割模型
- 图片实例分割（mask + box）
- 视频实例分割
- 模型训练（示例）
- 模型导出

## 0. 环境检查

In [ ]:
import torch
from ultralytics import YOLO

# 检查 PyTorch 和 CUDA
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 1. 加载预训练分割模型

首次运行会自动下载模型权重文件。

分割模型（`-seg` 后缀）输出像素级 mask + bounding box：
- `yolov8n-seg.pt` — Nano，最快（实时首选）
- `yolov8s-seg.pt` — Small
- `yolov8m-seg.pt` — Medium
- `yolov8l-seg.pt` — Large
- `yolov8x-seg.pt` — XLarge，最准

检测版（无 `-seg`）只输出 bounding box，没有 mask。

In [ ]:
model = YOLO("yolov8n-seg.pt")
print("模型加载完成！")
print(f"类别数: {model.model.nc}")
print(f"任务类型: 实例分割")

## 2. 图片实例分割

In [ ]:
# 使用预训练分割模型检测单张图片
# 替换为你的图片路径，或使用 Ultralytics 内置测试图片
results = model("https://ultralytics.com/images/bus.jpg", save=True, project="../outputs", name="notebook_test")

# 显示分割结果（含 mask）
results[0].show()

# 打印检测到的目标 + mask 面积
print(f"\n检测到 {len(results[0].boxes)} 个实例:\n")
for i, (box, mask) in enumerate(zip(results[0].boxes, results[0].masks)):
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    area = int(mask.data[0].sum())  # mask 像素数
    print(f"  [{i}] {model.names[cls_id]:20s} conf={conf:.3f}  mask={area}px")

## 3. 模型训练（演示）

使用 COCO128 小数据集快速演示训练流程。
**正式训练请使用 `scripts/train.py`**

In [ ]:
# 快速训练演示（仅 3 轮，验证流程）
model = YOLO("yolov8n-seg.pt")
results = model.train(
    data="coco128-seg.yaml",      # 内置小数据集（约 128 张图，含 mask 标注）
    epochs=3,                      # 仅演示用 3 轮
    imgsz=320,                     # 小尺寸加快速度
    batch=8,                       # 根据显存调整
    device=0,                      # 使用 GPU
)
print("训练演示完成！")

## 4. 模型导出

将模型导出为其他格式，便于部署。

In [ ]:
# 导出为 ONNX 格式（跨平台部署）
model.export(format="onnx", imgsz=640)
print("ONNX 导出完成！")

# 导出为 TensorRT 格式（NVIDIA GPU 加速）
# model.export(format="engine", imgsz=640, device=0)
# print("TensorRT 导出完成！")

## 5. 更多资源

- [Ultralytics 官方文档](https://docs.ultralytics.com/zh/)
- [YOLOv8 实例分割指南](https://docs.ultralytics.com/tasks/segment/)
- 运行 `python scripts/camera_detect.py --d455` 体验 D455 实时实例分割
- 运行 `python scripts/camera_detect.py` 体验普通摄像头实时分割